# Phutball per-puzzle curriculum (v3; expert iteration, Qwen3.5-9B thinking on, board + men list)

Per-puzzle version of curriculum_colab.ipynb (llm_train/curriculum_pp.py). v2 (rounds 1-9) allocated by bucket averages:
with 300 of 15,512 puzzles a round no puzzle came up twice, so parole never fired, and cheap stable-cut "NO WIN"
credit starved the no-win bucket. Here every puzzle has its own success rate from **8 samples**, counting only answers
the model **finished on its own**:
- **round 1 = screen**: 1,600 puzzles stratified over the 11 buckets, 8 samples each (also round 1's training data);
- **8/8 -> mastered**: never sampled again, and no new training examples (their banked examples are replayed so the
  from-base retrain does not forget them); **0/8 -> paroled**; spot checks (10% of each round) bring paroled puzzles
  back when the model finishes one correctly;
- **later rounds**: 150 puzzles x 8 samples (same cost as v2's 300 x 4): active puzzles by priority (1-(1-p)^8)(1-p) of
  their own rate, no repeats in consecutive rounds, at most 30% from one bucket; fresh puzzles refill the working set,
  weighted toward buckets whose screened puzzles turned out learnable;
- **training set**: each round's new examples + equal replay from the bank (each puzzle's latest verified examples),
  at most 1,000, fine-tuned from the base model (as before);
- **gate** unchanged (25 per group, 1 sample).

Starts from the v2 best (r6) and gates against its existing eval. **Runtime:** A100, High-RAM. **Cost:** round 1 (the
screen) ~4.5 h (~32 units); later rounds ~1 h (~7 units). Resumable: re-run all cells after a disconnect.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, subprocess, time, urllib.request
OLD = '/content/drive/MyDrive/phutball/curriculum'            # v2 run
OUT = '/content/drive/MyDrive/phutball/curriculum_pp'; os.makedirs(OUT, exist_ok=True)
INIT = f'{OLD}/r6/adapter/final'; INIT_EVAL = f'{OLD}/eval/g6'    # v2's best and its gate eval
POOL = 'llm_train/puzzles/pool_v2.jsonl.gz'; STATE = f'{OUT}/state.json'; BANK = f'{OUT}/bank.jsonl'; FMT = 'ascii+men'
CAP = 6144; SCREEN = 1600; N = 150; K = 8; ROUNDS = 15   # round 1 ~4.5 h, then ~1 h per round; raise ROUNDS any time (resumes)
PG = 25; EK = 4                                     # full eval (curves): 25 items per group x 4 samples
GATE_EK = 1; FULL_EVERY = 5                         # gate eval every round: 25 per group x 1 sample; full eval on round 1 and every 5th
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention
!pip -q uninstall -y torchao
!python -c "import trl, vllm, peft, transformers; print('trl', trl.__version__, 'vllm', vllm.__version__, 'peft', peft.__version__, 'transformers', transformers.__version__)"

In [ ]:
def serve(adapters: dict, max_len=CAP + 2048):
    !pkill -f "vllm serve" ; sleep 5
    extra = ['--enable-lora', '--max-lora-rank', '32', '--max-loras', str(max(1, len(adapters))), '--lora-modules',
             *[f'{k}={v}' for k, v in adapters.items()]] if adapters else []
    log = open('/content/vllm_serve.log', 'w')
    p = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3.5-9B', '--max-model-len', str(max_len), '--gpu-memory-utilization',
                          '0.90', '--port', '8000', *extra], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        if p.poll() is not None: break
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up:', list(adapters) or 'base'); return p
        except Exception: time.sleep(5)
    !tail -15 /content/vllm_serve.log
    raise RuntimeError('vLLM did not start')

EV = f'{OUT}/eval'; os.makedirs(EV, exist_ok=True)
def evaluate(name, model, samples=EK):
    if os.path.exists(f'{EV}/{name}-forced.summary.json'): print('done:', name); return
    !python -m llm_bench.run --model {model} --name {name} --prompt-format {FMT} --thinking true --qwen-thinking-sampling --samples {samples} --max-tokens {CAP} --per-group {PG} --concurrency 96 | tail -10
    !python -m llm_bench.run --model {model} --tokenizer Qwen/Qwen3.5-9B --prompt-format {FMT} --thinking true --salvage {name} --per-group {PG} --concurrency 48 | tail -10
    !cp llm_bench/results/{name}* {EV}/
def best_adapter():
    return (json.load(open(STATE)).get('best') if os.path.exists(STATE) else None) or INIT

## 1. Starting point

v2's best (r6) and its gate eval (`curriculum/eval/g6`) are the first 'best'; its full eval for the curves runs once here.

In [ ]:
if not os.path.exists(f'{EV}/r6v2-forced.summary.json'):
    server = serve({'r6v2': INIT}); evaluate('r6v2', 'r6v2'); !pkill -f "vllm serve"
else: print('start eval done')


## 2. Curriculum rounds (unattended)

In [ ]:
for R in range(1, ROUNDS + 1):
    d = f'{OUT}/r{R}'; os.makedirs(d, exist_ok=True)
    if not os.path.exists(f'{d}/ids.json'):
        if R == 1:
            !python -m llm_train.curriculum_pp screen --state {STATE} --pool {POOL} --n {SCREEN} --out {d}/ids.json | head -16
        else:
            !python -m llm_train.curriculum_pp allocate --state {STATE} --pool {POOL} --n {N} --round {R} --out {d}/ids.json | head -40
    if not os.path.exists(f'{d}/outcomes.jsonl'):
        print(f'=== round {R}: sampling from', best_adapter())
        !python -m llm_train.expert_iter --arm iid --adapter {best_adapter()} --pool {POOL} --ids {d}/ids.json --out {d} --k {K} --max-think {CAP} --seed {R} --prompt-format {FMT} --max-neg-ratio 1.0 2>&1 | grep --line-buffered -v -E "it/s\]|Processed prompts|Adding requests|INFO|WARNING"
    if not os.path.exists(f'{d}/train.jsonl'):
        !python -m llm_train.curriculum_pp update --state {STATE} --round {R} --outcomes {d}/outcomes.jsonl --sft {d}/sft.jsonl --bank {BANK} --train-out {d}/train.jsonl | tail -40
    if not os.path.exists(f'{d}/adapter/final/adapter_model.safetensors'):
        !python -m llm_train.sft_stop --data {d}/train.jsonl --out {d}/adapter --max-neg-ratio 1.0 2>&1 | grep --line-buffered -E "examples|'loss'|saved|rror|Traceback"
    full = (R == 1 or R % FULL_EVERY == 0)
    if not os.path.exists(f'{EV}/g{R}-forced.summary.json') or (full and not os.path.exists(f'{EV}/r{R}-forced.summary.json')):
        server = serve({f'r{R}': f'{d}/adapter/final'})
        evaluate(f'g{R}', f'r{R}', samples=GATE_EK)
        if full: evaluate(f'r{R}', f'r{R}')
        !pkill -f "vllm serve"
    if not os.path.exists(f'{d}/gate.txt'):
        !python -m llm_train.curriculum_pp gate --state {STATE} --round {R} --new {EV}/g{R} --adapter {d}/adapter/final --init-best {INIT_EVAL} --init-adapter {INIT} | tee {d}/gate.txt
print('curriculum done; best:', best_adapter())


## 3. Results

In [ ]:
names = ['r6v2'] + [f'r{R}' for R in range(1, ROUNDS + 1)]   # full evals: the v2 start, round 1 and every FULL_EVERY
keys = ('win: positive, 1-jump', 'win: positive, 2-jump', 'win: positive, 3-jump', 'win: positive, 4-jump',
        'win: near-miss negative', 'forced placement', 'block placement', 'prevent placement')
for label, suffix in (('FINISHED ON ITS OWN', '.summary.json'), ('BUDGET-FORCED', '-forced.summary.json')):
    S = {n: json.load(open(f'{EV}/{n}{suffix}')) for n in names if os.path.exists(f'{EV}/{n}{suffix}')}
    print(f"\n{label}: pass@1 / pass@{EK}  (ascii+men, thinking on, sampled, {CAP} tokens)")
    print(f"{'':26s}" + ''.join(f"{n:>14s}" for n in S))
    for k in keys:
        print(f"{k:26s}" + ''.join(f"{S[n][k]['accuracy']:>8.0%} /{S[n][k].get(f'pass@{EK}', float('nan')):>4.0%}" if k in S[n] else f"{'-':>14s}" for n in S))
    print(f"{'cut off (wins)':26s}" + ''.join(f"{S[n].get('win (all)', {}).get('truncated', 0):>14.0%}" for n in S))
    print(f"{'mean tokens':26s}" + ''.join(f"{S[n].get('mean_completion_tokens', 0):>14.0f}" for n in S))
if os.path.exists(STATE):
    st = json.load(open(STATE))
    print('\nscheduler log:')
    for e in st['log']:
        if e['event'] == 'gate': print(f"  round {e['round']}: {'PROMOTED' if e['promoted'] else 'kept best'} ({e['reason']})")
        elif e['event'] == 'update': print(f"  round {e['round']}: finished rates {e['finished_rates']} | {e['moves']} | {e['status']}")
        elif e['event'] == 'train_set': print(f"  round {e['round']}: train {e['new']} new + {e['replay']} replay ({e['dropped_mastered']} mastered dropped)")

In [ ]:
# optional: stop billing when everything above has finished (queue this cell last)
# from google.colab import runtime; runtime.unassign()